# 实验 14 — dlt 写入文件系统（Parquet / JSONL）

**目标**：换一个 destination，让 dlt 把数据写到本地文件系统的 Parquet 文件，模拟 **data lake / lakehouse staging layer** 的常见模式。

生产里一种常见架构是：
```
Source API → dlt → S3 (Parquet) → 引擎（Trino/Athena/DuckDB/Spark）→ dbt → 仓库表
```
用 filesystem destination 在本地试一遍这个模式，把数据落到 `data/lake/` 目录，再让 DuckDB / dbt 通过 `read_parquet()` 消费。

注意：dlt 的 filesystem destination 需要 `dlt[filesystem]` 或具体后端（如 `dlt[s3]`），本仓库的 `dlt[duckdb]` 已包含 fsspec 基础支持。

In [ ]:
# 准备：装 filesystem extras（如未装）
import subprocess
r = subprocess.run(['uv','pip','install','dlt[filesystem]','pyarrow'],
                   capture_output=True, text=True, cwd='..')
print(r.stdout[-500:])
if r.returncode != 0: print('STDERR:', r.stderr[-500:])

In [ ]:
# 把 daily_rates 写到本地 data/lake/raw_ecb_lake/daily_rates/*.parquet
import dlt, requests
from typing import Iterator
from pathlib import Path

lake = Path('../data/lake').resolve()
lake.mkdir(parents=True, exist_ok=True)

@dlt.resource(name='daily_rates', write_disposition='replace')
def daily_rates() -> Iterator[dict]:
    data = requests.get('https://api.frankfurter.app/2024-01-01..2024-01-31?to=USD,GBP', timeout=30).json()
    for d, rates in data['rates'].items():
        for cur, rate in rates.items():
            yield {'date': d, 'currency': cur, 'rate': rate}

pipe = dlt.pipeline(
    pipeline_name='ecb_lake',
    destination=dlt.destinations.filesystem(bucket_url=f'file://{lake}'),
    dataset_name='raw_ecb_lake',
)
print(pipe.run(daily_rates(), loader_file_format='parquet'))

In [ ]:
# 看 lake 目录的物理布局
import os
for root, dirs, files in os.walk('../data/lake'):
    rel = os.path.relpath(root, '../data/lake')
    for f in files:
        print(os.path.join(rel, f))

In [ ]:
# 用 DuckDB 直接读 parquet —— lakehouse 风格
import duckdb
con = duckdb.connect(':memory:')
con.execute("""
    select date, currency, rate
    from read_parquet('../data/lake/raw_ecb_lake/daily_rates/*.parquet')
    order by date, currency
    limit 10
""").df()

In [ ]:
# 在 dbt 里也能做：external table 或 dbt-duckdb 的 source-external 配置
# 这里手工等价演示——把 sandbox.duckdb 里也建一个 view 指向 parquet
import duckdb
con = duckdb.connect('../data/sandbox.duckdb')
con.execute("create or replace view raw_ecb_lake_view as select * from read_parquet('../data/lake/raw_ecb_lake/daily_rates/*.parquet')")
con.execute('select count(*) from raw_ecb_lake_view').fetchone()

## 生产环境踩坑提示

- **文件命名/分区**：默认按 load_id 切文件。Hive 风格分区要在 `layout` 配置：
  ```toml
  [destination.filesystem]
  layout = "{table_name}/year={year}/month={month}/{load_id}.{ext}"
  ```
- **`replace` 在 filesystem 上**：dlt 会先写新文件，再删旧文件。S3 这种 eventually consistent 的存储要小心读端看到中间态。
- **`merge` 在 filesystem 上需要后端支持**：纯文件系统不支持就地更新，dlt 会回退到 staging（写到临时区，再 rename）。Iceberg/Delta destination 才有真正的 ACID merge。
- **Parquet schema 演化**：Parquet 文件的 schema 是固定的，写新列后旧文件读出来缺这列。下游用 `union by name` 或带 schema 演化能力的引擎读。
- **生产化路径**：把 `bucket_url="s3://my-bucket/raw"` + AWS credentials 一替换就到了 S3。本仓库就是把它换成 `file://...`。

## 思考题

- 改成 `loader_file_format='jsonl'` 再跑，文件看起来什么样？什么场景用 JSONL 不用 Parquet？
- 用 hive-partition layout 跑一次，DuckDB 的 `read_parquet('.../year=*/month=*/*.parquet', hive_partitioning=true)` 能不能用上分区？
- 怎么让 dbt-duckdb 直接把 filesystem 上的 parquet 当作 source 表？（提示：`external` source meta）